# 01 — Auditoria dos dados experimentais

**Projeto:** Produção de bioetanol a partir do soro de leite
**Fonte:** relatório do projeto (`PROJETO_BIOETANOL_SORO_DE_LEITE_relatorio.docx`), Colégio IMP / Laboratório G-Óleo (UFLA), Lavras-MG, 2026.

## Objetivo deste notebook

Antes de qualquer modelagem técnica ou econômica, este notebook organiza **todos os dados experimentais disponíveis**, classifica cada um como `Medido`, `Calculado` (derivado dos medidos) ou `FALTANTE`, e documenta explicitamente as lacunas — sem tentar preenchê-las com suposições silenciosas.

Os dados brutos vêm de `data/raw/dados_experimentais.csv`, que foi extraído linha por linha do relatório do projeto (nunca de memória ou estimativa).


In [1]:
import sys
sys.path.insert(0, "../src")

import pandas as pd
import models as m

pd.set_option("display.max_colwidth", None)

df = m.load_experimental_data()
print(f"{len(df)} variáveis registradas.")
df

28 variáveis registradas.


,variavel,valor,unidade,origem,tipo,observacoes
0,volume_soro_cru,250,mL,Experimento (relatorio secao 3.4),Medido,Soro de leite fresco obtido em laticinio local
1,pH_soro_inicial,6.0,adimensional,Experimento (relatorio secao 3.4),Medido,NaN
2,pH_soro_acidificado,4.5-4.6,adimensional,Experimento (relatorio secao 3.4),Medido,Ponto isoeletrico das proteinas do soro
3,temperatura_desproteinizacao,90-100,C,Experimento (relatorio secao 3.4),Medido,Faixa de processo em banho-maria
4,tempo_aquecimento_desproteinizacao,15-30,min,Experimento (relatorio secao 3.4),Medido,Faixa de processo
5,volume_filtrado_pos_desproteinizacao,160,mL,Experimento (relatorio secao 3.4),Medido,Filtracao a vacuo (Buchner)
6,recuperacao_volume_filtracao,64.0,%,Experimento (relatorio secao 4.1 tabela 3),Calculado,160/250
7,massa_lactase_adicionada,4.0047,g,Experimento (relatorio secao 3.4),Medido,Balanca analitica resolucao 0.1 mg
8,atividade_enzimatica_estimada,106000,U.FCC,Experimento (relatorio secao 3.4),Calculado a partir do rotulo,10000 U.FCC/capsula x ~10.6 capsulas
9,temperatura_hidrolise,37-40,C,Experimento (relatorio secao 3.3),Medido,Faixa de trabalho da lactase


## Classificação por tipo de dado

- **Medido**: valor lido diretamente de um instrumento durante o experimento (balança, termômetro, proveta, cronômetro).
- **Calculado**: derivado aritmeticamente de valores medidos (ex.: um rendimento percentual).
- **FALTANTE**: variável que o processo científico exigiria, mas que não foi determinada nesta execução.


In [2]:
contagem = df["tipo"].value_counts()
contagem

tipo
Medido                              16
Calculado                            4
FALTANTE                             2
Premissa de projeto (nao medida)     2
Nao medido                           2
Calculado a partir do rotulo         1
Medido (qualitativo)                 1
Name: count, dtype: int64

In [3]:
faltantes = df[df["tipo"].str.contains("FALTANTE", case=False, na=False)]
faltantes[["variavel", "valor", "unidade", "observacoes"]]

,variavel,valor,unidade,observacoes
20,densidade_destilado,NAO_DETERMINADA,g/mL,Nao medida nesta execucao
21,teor_alcoolico_destilado,NAO_DETERMINADO,% v/v ou GL,Depende da densidade nao medida


## ⚠️ Lacuna crítica: teor alcoólico / densidade do destilado

O experimento produziu **15 mL de destilado** a partir de 250 mL de soro, com evidências qualitativas fortes de que se trata de uma solução hidroalcoólica (temperatura de coleta ≈78 °C, coincidente com o ponto de ebulição do etanol; combustão positiva; chama sem coloração visível — característica do etanol, não de hidrocarbonetos mais pesados).

**Mas a densidade do destilado nunca foi medida.** Sem densidade, não há como calcular o teor alcoólico real, e sem teor alcoólico não sabemos quantos mL (ou gramas) de etanol *puro* foram de fato produzidos.

**Decisão adotada para este projeto** (confirmada com o responsável): usar a faixa que o **próprio relatório do projeto já projetava antes da execução** (seção 4.6 — "Comparação com os resultados esperados"): **5% a 12% v/v, antes de qualquer retificação**. Essa faixa é tratada como uma **premissa de projeto**, não como um dado medido, e alimenta três cenários (conservador/experimental/otimista) no notebook `03_technical_model`.


## Balanço de volumes (dados medidos, recalculados aqui para conferência)

Reproduzimos os percentuais que aparecem no relatório a partir dos volumes brutos medidos, para confirmar que não há erro aritmético antes de usá-los como base do modelo.


In [4]:
soro_cru_mL = 250
filtrado_mL = 160
mosto_mL = 160
destilado_mL = 15

balanco = pd.DataFrame({
    "etapa": ["Soro de leite cru", "Filtrado (pós-desproteinização)", "Mosto submetido à destilação", "Destilado obtido"],
    "volume_mL": [soro_cru_mL, filtrado_mL, mosto_mL, destilado_mL],
})
balanco["% do soro inicial"] = (balanco["volume_mL"] / soro_cru_mL * 100).round(1)
balanco

,etapa,volume_mL,% do soro inicial
0,Soro de leite cru,250,100.0
1,Filtrado (pós-desproteinização),160,64.0
2,Mosto submetido à destilação,160,64.0
3,Destilado obtido,15,6.0


In [5]:
recuperacao_filtracao = filtrado_mL / soro_cru_mL
razao_destilado_mosto = destilado_mL / mosto_mL
razao_destilado_soro = destilado_mL / soro_cru_mL

print(f"Recuperação na filtração:      {recuperacao_filtracao:.1%}  (relatório: 64,0%)")
print(f"Destilado / mosto:             {razao_destilado_mosto:.1%}  (relatório: 9,4%)")
print(f"Destilado / soro inicial:      {razao_destilado_soro:.1%}  (relatório: 6,0%)")
print(f"Destilado por litro de soro:   {razao_destilado_soro*1000:.0f} mL/L (relatório: 60 mL/L)")

assert abs(recuperacao_filtracao - 0.64) < 0.001
assert abs(razao_destilado_soro - 0.06) < 0.001
print("\nConferido: os percentuais batem com os valores citados no relatório.")

Recuperação na filtração:      64.0%  (relatório: 64,0%)
Destilado / mosto:             9.4%  (relatório: 9,4%)
Destilado / soro inicial:      6.0%  (relatório: 6,0%)
Destilado por litro de soro:   60 mL/L (relatório: 60 mL/L)

Conferido: os percentuais batem com os valores citados no relatório.


## Outras limitações registradas no relatório (seção 4.5)

1. **Sem caracterização quantitativa do produto** (densidade/teor alcoólico) — já discutido acima.
2. **Sem monitoramento cinético da fermentação** — não houve contagem de bolhas nem registro periódico ao longo das 72 h; a curva de atividade fermentativa não pôde ser construída.
3. **Sem ensaio controle** (sem lactase) — não é possível isolar, por comparação direta, o efeito da hidrólise enzimática.
4. **Execução única, sem réplicas** — impede qualquer avaliação de reprodutibilidade/desvio-padrão do rendimento obtido.

Essas limitações não invalidam o resultado (a inflamabilidade do destilado é evidência qualitativa real), mas **limitam o que os notebooks seguintes podem afirmar com base neles**: o modelo técnico e econômico vai propagar essa incerteza através de cenários explícitos, não escondê-la atrás de um número único.


## Inconsistências / pontos a esclarecer

- **Fermento biológico**: o relatório cita "2 pacotes (4 g utilizados)" sem informar o tamanho de cada pacote — não é possível saber se os 2 pacotes foram usados integralmente ou parcialmente. Não afeta os cálculos (usamos a massa efetivamente utilizada, 4 g, e o custo efetivamente gasto, R$ 5,00), mas fica registrado como ambiguidade da fonte.
- **Custo do soro de leite**: não contabilizado no relatório (obtido "gratuitamente" em laticínio local). Para o modelo econômico industrial (notebook 04), isso não pode ser assumido em escala — precisaremos de um preço de mercado ou custo de logística, que não temos; o modelo econômico deste projeto **não inclui custo de aquisição do soro**, e essa omissão é declarada explicitamente nas conclusões.


## Resumo da auditoria (Etapa 1)

| Categoria | Situação |
|---|---|
| Volumes do processo (soro → filtrado → mosto → destilado) | ✅ Medidos, consistentes |
| Massas de insumos (lactase, fermento) | ✅ Medidas (balança analítica) |
| Temperaturas e tempos de cada etapa | ✅ Medidos (como faixas de processo) |
| Teste de chama | ✅ Medido (qualitativo) |
| Densidade / teor alcoólico do destilado | ❌ Faltante — tratado como premissa de projeto (5–12% v/v) |
| Concentração de lactose no soro usado | ❌ Faltante — será usada uma concentração de literatura como proxy (notebook 02) |
| Cinética de fermentação | ❌ Faltante — não afeta o balanço final, mas limita a compreensão do processo |
| Custo de reagentes | ✅ Medido, mas em preço de **varejo** (não industrial) |
| Custo do soro / equipamentos | ❌ Não contabilizado |

Com esse diagnóstico, seguimos para o notebook `02_external_data`, que documenta as fontes externas usadas para preencher as lacunas que **podem** ser preenchidas de forma defensável (proxies de literatura, com fonte e ressalva), e para deixar claro o que continua em aberto.
